In [1]:
import pandas as pd
import numpy as np
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
print(train.shape, test.shape)

(891, 12) (418, 11)


In [2]:
for df in [train, test]:
    # 名前から敬称を抽出（Mr/Mrs/Miss など）
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(
        ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    
    # 家族の人数
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    # 欠損値処理
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Embarked'] = df['Embarked'].fillna('S')
    
    # 数値変換
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Title'] = df['Title'].map(
        {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})
    df['Title'] = df['Title'].fillna(0)
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

print("特徴量エンジニアリング完了！")

特徴量エンジニアリング完了！


In [3]:
from sklearn.ensemble import RandomForestClassifier

features = ['Pclass', 'Sex', 'Age', 'Fare', 
            'FamilySize', 'IsAlone', 'Title', 'Embarked']

X_train = train[features]
y_train = train['Survived']
X_test = test[features]

model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
model.fit(X_train, y_train)
print(f"訓練精度: {model.score(X_train, y_train):.2%}")

訓練精度: 86.31%


In [4]:
predictions = model.predict(X_test)
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})
submission.to_csv('submission.csv', index=False)
print("完成！")
print(submission.head())

完成！
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
